In [26]:
# Params
csv_path = "/Users/cooperfoster/Desktop/hoops/brk_repl2.csv"          # CSV with column 'game_id'
out_dir   = "/Users/cooperfoster/Desktop/hoops/brk dirty"        # folder for *.txt
max_per_minute = 6                          # hard cap

# ---- Code ----
import pandas as pd
from pathlib import Path
from time import sleep, monotonic

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

URL_TPL = "https://www.basketball-reference.com/boxscores/shot-chart/{gid}.html"
DELAY = 60.0 / max_per_minute  # seconds between requests

# I/O
ids = pd.read_csv(csv_path)["game_id"].astype(str).tolist()
Path(out_dir).mkdir(parents=True, exist_ok=True)

# Selenium
opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("--disable-gpu")
opts.add_argument("--no-sandbox")
opts.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
opts.add_experimental_option("excludeSwitches", ["enable-automation"])
opts.add_experimental_option("useAutomationExtension", False)
driver = webdriver.Chrome(options=opts)

start = monotonic()
last_req = start
done = 0
skipped = 0

try:
    for gid in ids:
        out_file = Path(out_dir) / f"{gid}.txt"
        if out_file.exists():
            skipped += 1
            continue

        # rate limit
        elapsed = monotonic() - last_req
        if elapsed < DELAY:
            sleep(DELAY - elapsed)

        url = URL_TPL.format(gid=gid)
        try:
            driver.get(url)
            html = driver.page_source
            out_file.write_text(html, encoding="utf-8")
            done += 1
            last_req = monotonic()
            print(f"saved {gid}")
        except Exception as e:
            # simple retry once after delay
            sleep(DELAY)
            try:
                driver.get(url)
                html = driver.page_source
                out_file.write_text(html, encoding="utf-8")
                done += 1
                last_req = monotonic()
                print(f"saved {gid} (retry)")
            except Exception as e2:
                print(f"failed {gid}: {e2}")

finally:
    driver.quit()
    total_s = monotonic() - start
    print(f"completed. saved={done}, skipped_existing={skipped}, time_sec={int(total_s)}")

saved 202501040BRK
saved 202502120BRK
completed. saved=2, skipped_existing=8, time_sec=107


In [2]:
import pandas as pd
from pathlib import Path

# Inputs
csv_path = "/Users/cooperfoster/Desktop/hoops/game_log.csv"       # original CSV with game_id column
out_dir  = "/Users/cooperfoster/Desktop/hoops/shot chart txts"      # folder with saved .txt files
out_csv  = "/Users/cooperfoster/Desktop/hoopsmissing_game_ids.csv"

# Load full list
ids = pd.read_csv(csv_path)["game_id"].astype(str).tolist()

# Collect saved game_ids from txt filenames
saved_files = list(Path(out_dir).glob("*.txt"))
saved_ids = {f.stem for f in saved_files}   # stem = filename without extension

# Find missing
missing_ids = [gid for gid in ids if gid not in saved_ids]

# Save to csv
pd.DataFrame({"game_id": missing_ids}).to_csv(out_csv, index=False)

print(f"Missing count: {len(missing_ids)}")
print(f"CSV written to {out_csv}")

Missing count: 4
CSV written to /Users/cooperfoster/Desktop/hoopsmissing_game_ids.csv


In [6]:
import pandas as pd
from pathlib import Path
from collections import Counter

# Inputs
codes_csv = "/users/cooperfoster/Desktop/hoops/team_codes.csv"    # your lookup table
out_dir   = "/users/cooperfoster/Desktop/hoops/shot chart txts/"      # folder with .txt files

# Load team codes
codes_df = pd.read_csv(codes_csv)
# Assuming there's a column named 'team_code' in the CSV
codes = codes_df['code'].astype(str).tolist()

# Gather filenames
files = list(Path(out_dir).glob("*.txt"))
file_names = [f.name for f in files]

# Count matches by team code in file name
counts = Counter()
for fname in file_names:
    for code in codes:
        if code in fname:
            counts[code] += 1

# Convert to dataframe
counts_df = pd.DataFrame(list(counts.items()), columns=['code', 'file_count'])

# Merge with lookup for completeness (teams with zero files show up as 0)
result = codes_df.merge(counts_df, on="code", how="left").fillna(0)
result['file_count'] = result['file_count'].astype(int)

# Output
print(result)

                      team code  file_count
0            Atlanta Hawks  ATL          40
1            Brooklyn Nets  BKN          41
2           Boston Celtics  BOS          41
3            Chicago Bulls  CHI          41
4        Charlotte Hornets  CHO          41
5      Cleveland Cavaliers  CLE          41
6         Dallas Mavericks  DAL          40
7           Denver Nuggets  DEN          41
8          Detroit Pistons  DET          41
9    Golden State Warriors  GSW          41
10         Houston Rockets  HOU          41
11          Indiana Pacers  IND          41
12    Los Angeles Clippers  LAC          41
13      Los Angeles Lakers  LAL          41
14       Memphis Grizzlies  MEM          41
15              Miami Heat  MIA          41
16         Milwaukee Bucks  MIL          42
17  Minnesota Timberwolves  MIN          41
18    New Orleans Pelicans  NOP          41
19         New York Knicks  NYK          41
20   Oklahoma City Thunder  OKC          43
21           Orlando Magic  ORL 

In [7]:
print(sum(result['file_count']))

1231


In [28]:
from pathlib import Path
import re

# --- configure ---
input_dir  = Path("/users/cooperfoster/desktop/hoops/shot chart txts")    # folder containing original txt files
output_dir = Path("/users/cooperfoster/desktop/hoops/clean txts") # folder for cleaned copies
output_dir.mkdir(parents=True, exist_ok=True)

# NBA team codes (3-letter common codes; extend if your source uses alternates)
TEAM_CODES = {
    "ATL","BOS","BRK","CHA","CHI","CLE","DAL","DEN","DET","GSW","HOU","IND","LAC",
    "LAL","MEM","MIA","MIL","MIN","NOP","NYK","OKC","ORL","PHI","PHX","POR","SAC",
    "SAS","TOR","UTA","WAS"
}

# team header detector: keep short HTML/text lines that contain a team code token
team_header_rx = re.compile(
    r"(?:>|\b)(%s)(?:\b|<|\))" % "|".join(sorted(TEAM_CODES)),
    flags=re.IGNORECASE
)

# shot marker detector: matches exactly the marker shown in your sample
# allows any quarter/tip text; captures top/left pixel coords if needed
shot_rx = re.compile(
    r'<div\s+style="top:\s*(\d+)px;left:\s*(\d+)px;"\s+tip="[^"]*"\s+class="tooltip[^"]*">\s*×\s*</div>',
    flags=re.IGNORECASE
)



In [30]:
from pathlib import Path
from bs4 import BeautifulSoup

def clean_shot_logs(input_dir, output_dir):
    in_path = Path(input_dir)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    for txt_file in in_path.glob("*.txt"):
        html = txt_file.read_text(encoding="utf-8", errors="ignore")
        soup = BeautifulSoup(html, "html.parser")

        # find both shot sections
        shot_blocks = soup.select('div.shot-area[id^="shots-"]')
        if not shot_blocks:
            # nothing to do; write a minimal file so downstream code can detect it
            (out_path / txt_file.name).write_text("<!-- NO SHOT BLOCKS FOUND -->\n", encoding="utf-8")
            continue

        out_lines = ["<!-- CLEANED SHOT LOG -->"]
        for i, shots_div in enumerate(shot_blocks, start=1):
            # team code from id="shots-XXX"
            shots_id = shots_div.get("id", "")
            team_code = shots_id.split("-", 1)[1] if "-" in shots_id else "UNK"

            out_lines.append(f"<!-- SECTION {i} START -->")
            out_lines.append(f'<div id="wrapper-{team_code}">')
            out_lines.append(f'<div id="{shots_id}" class="shot-area">')

            # keep every shot div; do not filter on glyph
            for shot in shots_div.find_all("div", class_="tooltip"):
                # write the original HTML of each observation
                out_lines.append(str(shot))

            # close the two wrappers we opened
            out_lines.append("</div>")   # close shots-TEAM
            out_lines.append("</div>")   # close wrapper-TEAM
            out_lines.append(f"<!-- SECTION {i} END -->")

        cleaned = "\n".join(out_lines) + "\n"

        # write to new folder; never overwrite originals
        (out_path / txt_file.name).write_text(cleaned, encoding="utf-8")

if __name__ == "__main__":
    # example:
    # clean_shot_logs("path/to/original_txts", "path/to/cleaned_txts")
    pass

In [31]:
clean_shot_logs("/users/cooperfoster/desktop/hoops/shot chart txts", "/users/cooperfoster/desktop/hoops/clean txts")

In [32]:
from pathlib import Path
import re, csv

# --- configure ---
in_dir  = Path("/users/cooperfoster/desktop/hoops/clean txts")   # folder with CLEANED txts
out_dir = Path("/users/cooperfoster/desktop/hoops/csv1")      # folder for per-game CSVs
out_dir.mkdir(parents=True, exist_ok=True)

# section extractor: team code + inner HTML of the shot block
sec_rx = re.compile(
    r"<!-- SECTION \d+ START -->\s*<div id=\"wrapper-([A-Z]{2,4})\">"
    r"\s*<div id=\"shots-\1\" class=\"shot-area\">(.*?)</div>\s*</div>\s*<!-- SECTION \d+ END -->",
    re.DOTALL
)

# shot div extractor: keep the original HTML for the row
shot_rx = re.compile(r"<div\b[^>]*\bclass=\"tooltip[^>]*>.*?</div>", re.DOTALL)

def txt_to_rows(txt: str, game_id: str):
    sections = sec_rx.findall(txt)
    rows = []
    if not sections:
        return rows
    # map sections to team codes and shot HTMLs
    teams = [t for t, _ in sections]
    for idx, (team, block_html) in enumerate(sections):
        opp = next((t for i, t in enumerate(teams) if i != idx), "UNK") if len(teams) >= 2 else "UNK"
        for m in shot_rx.finditer(block_html):
            rows.append({
                "shot_txt": m.group(0).strip(),
                "team": team,
                "opp": opp,
                "game_id": game_id
            })
    return rows

def process_folder(in_dir: Path, out_dir: Path):
    for p in sorted(in_dir.glob("*.txt")):
        game_id = p.stem
        txt = p.read_text(encoding="utf-8", errors="ignore")
        rows = txt_to_rows(txt, game_id)
        out_p = out_dir / f"{game_id}.csv"
        with out_p.open("w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["shot_txt","team","opp","game_id"])
            w.writeheader()
            for r in rows:
                w.writerow(r)
        print(f"Wrote {out_p.name} with {len(rows)} rows")

process_folder(in_dir, out_dir)

Wrote 202410220BOS.csv with 173 rows
Wrote 202410220LAL.csv with 180 rows
Wrote 202410230ATL.csv with 170 rows
Wrote 202410230DET.csv with 171 rows
Wrote 202410230HOU.csv with 187 rows
Wrote 202410230LAC.csv with 170 rows
Wrote 202410230MIA.csv with 178 rows
Wrote 202410230NOP.csv with 182 rows
Wrote 202410230PHI.csv with 178 rows
Wrote 202410230POR.csv with 185 rows
Wrote 202410230TOR.csv with 172 rows
Wrote 202410230UTA.csv with 175 rows
Wrote 202410240DAL.csv with 188 rows
Wrote 202410240DEN.csv with 198 rows
Wrote 202410240SAC.csv with 166 rows
Wrote 202410240WAS.csv with 185 rows
Wrote 202410250ATL.csv with 176 rows
Wrote 202410250CLE.csv with 163 rows
Wrote 202410250HOU.csv with 189 rows
Wrote 202410250LAL.csv with 157 rows
Wrote 202410250MIL.csv with 189 rows
Wrote 202410250NYK.csv with 166 rows
Wrote 202410250ORL.csv with 158 rows
Wrote 202410250POR.csv with 176 rows
Wrote 202410250TOR.csv with 152 rows
Wrote 202410250UTA.csv with 197 rows
Wrote 202410260CHI.csv with 199 rows
W

In [33]:
from pathlib import Path
import csv, re, html

# --- configure ---
in_dir  = Path("/users/cooperfoster/desktop/hoops/csv1")    # folder containing CSVs with columns: shot_txt, team, opp, game_id
out_dir = Path("/users/cooperfoster/desktop/hoops/csv2")# folder for parsed CSVs
out_dir.mkdir(parents=True, exist_ok=True)

# --- regexes ---
STYLE_RX   = re.compile(r'style="top:\s*(-?\d+)px;left:\s*(-?\d+)px;"', re.I)
TIP_RX     = re.compile(r'tip="(.*?)"', re.I | re.S)
RESULT_CLS = re.compile(r'class="[^"]*\b(miss|made)\b', re.I)

# helpers
ORD_Q = re.compile(r'(\d+)(?:st|nd|rd|th)\s+quarter', re.I)
OT_Q  = re.compile(r'(\d+)OT|\bOT\b|\bovertime\b', re.I)
TIME_RX = re.compile(r'(\d{1,2}:\d{2}(?:\.\d+)?)\s*remaining', re.I)
SHOT_LINE = re.compile(r'<br>\s*([^<]+?)\s+(made|missed)\s+([23])\-pointer\s+from\s+(\d+)\s*ft', re.I)
DIST_FALLBACK = re.compile(r'from\s+(\d+)\s*ft', re.I)
SHOTTYPE_FALLBACK = re.compile(r'\b([23])\-pointer\b', re.I)
SHOOTER_FALLBACK  = re.compile(r'<br>\s*([^<]+?)\s+(?:made|missed)\b', re.I)

def parse_row(raw_html: str):
    # top/left
    m = STYLE_RX.search(raw_html)
    if m:
        top_val = int(m.group(1))
        left_val = int(m.group(2))
        # replace negatives with 0
        top = str(top_val if top_val >= 0 else 0)
        left = str(left_val if left_val >= 0 else 0)
    else:
        top, left = None, None

    # tip text
    tip_m = TIP_RX.search(raw_html)
    tip_raw = tip_m.group(1) if tip_m else ""
    tip_txt = html.unescape(tip_raw)  # converts &lt;br&gt; -> <br>, etc.
    # quarter
    q = None
    q1 = ORD_Q.search(tip_txt)
    if q1:
        q = q1.group(1)  # "1","2","3","4"
    else:
        q2 = OT_Q.search(tip_txt)
        if q2 and q2.group(1):
            q = f"{q2.group(1)}OT"
        elif q2:
            q = "OT"
    # time
    t = None
    t1 = TIME_RX.search(tip_txt)
    if t1:
        t = t1.group(1)

    # shooter, result, shot_type, distance
    shooter = result = shot_type = distance = None
    s = SHOT_LINE.search(tip_txt)
    if s:
        shooter, result, shot_type, distance = s.group(1).strip(), s.group(2).lower(), s.group(3), s.group(4)
    else:
        # fallbacks
        # result from class or glyph
        r = RESULT_CLS.search(raw_html)
        if r:
            result = r.group(1).lower()
        elif "×" in raw_html:
            result = "missed"
        elif "●" in raw_html:
            result = "made"

        st = SHOTTYPE_FALLBACK.search(tip_txt)
        if st:
            shot_type = st.group(1)

        d = DIST_FALLBACK.search(tip_txt)
        if d:
            distance = d.group(1)

        sh = SHOOTER_FALLBACK.search(tip_txt)
        if sh:
            shooter = sh.group(1).strip()

    return {
        "top": top,
        "left": left,
        "quarter": q,
        "time": t,
        "shooter": shooter,
        "shot_type": "3" if shot_type == "3" else ("2" if shot_type == "2" else None),
        "result": result,
        "distance": distance
    }

def process_file(p_in: Path, p_out: Path):
    with p_in.open("r", encoding="utf-8", newline="") as f_in, \
         p_out.open("w", encoding="utf-8", newline="") as f_out:
        r = csv.DictReader(f_in)
        w = csv.DictWriter(f_out, fieldnames=[
            "top","left","quarter","time","shooter","shot_type","result","distance",
            "team","opp","game_id"
        ])
        w.writeheader()
        for row in r:
            parsed = parse_row(row["shot_txt"])
            parsed.update({
                "team": row.get("team"),
                "opp": row.get("opp"),
                "game_id": row.get("game_id"),
            })
            w.writerow(parsed)

def process_folder(in_dir: Path, out_dir: Path):
    for p in sorted(in_dir.glob("*.csv")):
        out_p = out_dir / p.name
        process_file(p, out_p)
        print(f"Wrote {out_p.name}")

process_folder(in_dir, out_dir)

Wrote 202410220BOS.csv
Wrote 202410220LAL.csv
Wrote 202410230ATL.csv
Wrote 202410230DET.csv
Wrote 202410230HOU.csv
Wrote 202410230LAC.csv
Wrote 202410230MIA.csv
Wrote 202410230NOP.csv
Wrote 202410230PHI.csv
Wrote 202410230POR.csv
Wrote 202410230TOR.csv
Wrote 202410230UTA.csv
Wrote 202410240DAL.csv
Wrote 202410240DEN.csv
Wrote 202410240SAC.csv
Wrote 202410240WAS.csv
Wrote 202410250ATL.csv
Wrote 202410250CLE.csv
Wrote 202410250HOU.csv
Wrote 202410250LAL.csv
Wrote 202410250MIL.csv
Wrote 202410250NYK.csv
Wrote 202410250ORL.csv
Wrote 202410250POR.csv
Wrote 202410250TOR.csv
Wrote 202410250UTA.csv
Wrote 202410260CHI.csv
Wrote 202410260CHO.csv
Wrote 202410260DEN.csv
Wrote 202410260DET.csv
Wrote 202410260LAL.csv
Wrote 202410260MEM.csv
Wrote 202410260MIN.csv
Wrote 202410260PHO.csv
Wrote 202410260SAS.csv
Wrote 202410260WAS.csv
Wrote 202410270BRK.csv
Wrote 202410270GSW.csv
Wrote 202410270IND.csv
Wrote 202410270OKC.csv
Wrote 202410270POR.csv
Wrote 202410280ATL.csv
Wrote 202410280BOS.csv
Wrote 20241

In [34]:
import pandas as pd
from pathlib import Path

# --- configure ---
in_dir  = Path("/users/cooperfoster/desktop/hoops/csv2")     # folder with per-game parsed CSVs
out_file = Path("/users/cooperfoster/desktop/hoops/all_shots.csv") # final rollup CSV

# read and bind all
dfs = []
for f in sorted(in_dir.glob("*.csv")):
    try:
        df = pd.read_csv(f, dtype=str)  # keep everything as string to avoid coercion issues
        dfs.append(df)
    except Exception as e:
        print(f"Skipping {f.name}: {e}")

if dfs:
    final = pd.concat(dfs, ignore_index=True)
    final.to_csv(out_file, index=False)
    print(f"Wrote {out_file} with {len(final)} rows from {len(dfs)} files")
else:
    print("No CSVs found")

Wrote /users/cooperfoster/desktop/hoops/all_shots.csv with 218904 rows from 1231 files


In [18]:
import pandas as pd
from pathlib import Path
import csv

# --- configure ---
in_dir  = Path("/users/cooperfoster/desktop/hoops/csv2")          # folder with structured/parsed CSVs
out_file = Path("/users/cooperfoster/desktop/hoops/few_obs_games.csv")  # output list of game_ids

bad_ids = []

for f in sorted(in_dir.glob("*.csv")):
    try:
        df = pd.read_csv(f, dtype=str)
        if len(df) < 50:
            # assume all rows in a file have same game_id
            gid = df["game_id"].iloc[0] if not df.empty else f.stem
            bad_ids.append({"game_id": gid, "n_obs": len(df)})
    except Exception as e:
        print(f"Skipping {f.name}: {e}")

# write output CSV
with out_file.open("w", newline="", encoding="utf-8") as out:
    w = csv.DictWriter(out, fieldnames=["game_id","n_obs"])
    w.writeheader()
    w.writerows(bad_ids)

print(f"Wrote {out_file} with {len(bad_ids)} game_ids < 5 obs")

Wrote /users/cooperfoster/desktop/hoops/few_obs_games.csv with 42 game_ids < 5 obs


In [19]:
import pandas as pd
from pathlib import Path

# --- configure ---
game_log_file = Path("/users/cooperfoster/desktop/hoops/game_log.csv")
fixed_file    = Path("/users/cooperfoster/desktop/hoops/game_log_fixed.csv")
brk_ids_file  = Path("/users/cooperfoster/desktop/hoops/brk_game_ids.csv")

# read
df = pd.read_csv(game_log_file, dtype=str)

# replace codes
df = df.replace("BKN", "BRK")

# save fixed version
df.to_csv(fixed_file, index=False)
print(f"Saved corrected log -> {fixed_file}")

# extract BRK game_ids
# assume game_id column exists
brk_df = df[df.apply(lambda row: row.astype(str).str.contains("BRK").any(), axis=1)]

brk_df[["game_id"]].drop_duplicates().to_csv(brk_ids_file, index=False)
print(f"Saved BRK game_ids -> {brk_ids_file} ({len(brk_df)} rows)")

Saved corrected log -> /users/cooperfoster/desktop/hoops/game_log_fixed.csv
Saved BRK game_ids -> /users/cooperfoster/desktop/hoops/brk_game_ids.csv (82 rows)


In [25]:
import pandas as pd
from pathlib import Path
import csv

# --- configure ---
game_ids_file = Path("/users/cooperfoster/desktop/hoops/brk_repl.csv")    # input list of game_ids
check_dir     = Path("/users/cooperfoster/desktop/hoops/brk dirty") # folder with files (like parsed CSVs)
out_file      = Path("/users/cooperfoster/desktop/hoops/brk_repl3.csv")

# load ids
df = pd.read_csv(game_ids_file, dtype=str)
ids = df["game_id"].astype(str).str.strip()
ids = ids[ids.ne("")].drop_duplicates().tolist()

missing = []
for gid in ids:
    if not (check_dir / f"{gid}.txt").exists():
        missing.append({"game_id": gid})

# write results
with out_file.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["game_id"])
    w.writeheader()
    w.writerows(missing)

print(f"Checked {len(ids)} IDs. Missing: {len(missing)}. Output -> {out_file}")

Checked 41 IDs. Missing: 2. Output -> /users/cooperfoster/desktop/hoops/brk_repl3.csv


In [27]:
from pathlib import Path

# --- configure ---
target_dir = Path("/users/cooperfoster/desktop/hoops/shot chart txts")  # folder to clean

for f in target_dir.glob("*BKN*"):
    if f.is_file():
        try:
            f.unlink()
            print(f"Deleted: {f.name}")
        except Exception as e:
            print(f"Error deleting {f.name}: {e}")

print("Done")

Deleted: 202502100BKN.txt
Deleted: 202501040BKN.txt
Deleted: 202504080BKN.txt
Deleted: 202411040BKN.txt
Deleted: 202501220BKN.txt
Deleted: 202502260BKN.txt
Deleted: 202504100BKN.txt
Deleted: 202412010BKN.txt
Deleted: 202410290BKN.txt
Deleted: 202412270BKN.txt
Deleted: 202412080BKN.txt
Deleted: 202503240BKN.txt
Deleted: 202503100BKN.txt
Deleted: 202501210BKN.txt
Deleted: 202503260BKN.txt
Deleted: 202501060BKN.txt
Deleted: 202502120BKN.txt
Deleted: 202411290BKN.txt
Deleted: 202504030BKN.txt
Deleted: 202504130BKN.txt
Deleted: 202501250BKN.txt
Deleted: 202411030BKN.txt
Deleted: 202411130BKN.txt
Deleted: 202504060BKN.txt
Deleted: 202502280BKN.txt
Deleted: 202502070BKN.txt
Deleted: 202412210BKN.txt
Deleted: 202412160BKN.txt
Deleted: 202502200BKN.txt
Deleted: 202410270BKN.txt
Deleted: 202503150BKN.txt
Deleted: 202411190BKN.txt
Deleted: 202502040BKN.txt
Deleted: 202503280BKN.txt
Deleted: 202412040BKN.txt
Deleted: 202411010BKN.txt
Deleted: 202501270BKN.txt
Deleted: 202501080BKN.txt
Deleted: 202